# Pitch to Postgres: Designing and Querying a Relational Database for FIFA World Cup Data
## Milestone 4: Analytical queries (`04_queries.ipynb`)

**DS604 Introduction to Data Management.** Group: Devanshi Dudhatra (202618027), Pujita Sunapu (202618003), Rahul Saha (202618037)

**Data:** Joshua C. Fjelstul, *The Fjelstul World Cup Database*, <https://github.com/jfjelstul/worldcup>, CC-BY-SA 4.0. Coverage: the 21 men's World Cups from 1930 to 2018. The data was loaded by `03_load_data.ipynb`; the modifications are listed there.

This notebook runs the 30 queries in `sql/queries.sql`. For each query it shows the business question, the SQL concepts used, the expected shape of the result, the SQL itself, the result from `pd.read_sql_query` (first 10 rows) and an interpretation. Section 9 then validates results against well-known World Cup facts and independent pandas calculations on the raw CSV files.

**Conventions**
- Wins, draws and losses come from the score, so a match settled by a penalty shoot-out counts as a **draw** (FIFA's convention).
- Historical teams (West Germany, Soviet Union, ...) are separate from their successors unless stated otherwise.
- Cards, line-ups and substitutions exist from 1970; shoot-outs from 1982.

This copy was executed against the **local** test database. Changing only `.env` runs it on the course server.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import db                                  # SQLAlchemy engine from .env

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)

engine = db.get_engine()
print("Target database:", db.describe())     # no password is ever printed
R = {}                                       # keeps every result for the validation section

Target database: postgres@localhost:5432/worldcup  schema: (default search_path)


## 1. Team performance

### Q01. Most successful teams

**Business question:** Which teams are the most successful, counting titles, final appearances (top two finishes) and top-four finishes?

**SQL concepts:** INNER JOIN, GROUP BY, conditional aggregation SUM(CASE ...), multi-column ORDER BY

**Expected result:** one row per team that ever finished in the top four (25 rows): team_name, titles, runners_up, finals (top two), top_four. Brazil first with 5 titles. Note: 1950 had no final match; its top two (Uruguay, Brazil) come from the final round group.

In [2]:
q01 = """
SELECT t.team_name,
       SUM(CASE WHEN s.position = 1 THEN 1 ELSE 0 END)  AS titles,
       SUM(CASE WHEN s.position = 2 THEN 1 ELSE 0 END)  AS runners_up,
       SUM(CASE WHEN s.position <= 2 THEN 1 ELSE 0 END) AS finals,
       COUNT(*)                                         AS top_four
FROM tournament_standings s
JOIN teams t ON t.team_id = s.team_id
GROUP BY t.team_name
ORDER BY titles DESC, finals DESC, top_four DESC, t.team_name;
"""
df = pd.read_sql_query(q01, engine)
R["Q01"] = df
print(df.head(10))

      team_name  titles  runners_up  finals  top_four
0        Brazil       5           2       7        11
1         Italy       4           2       6         8
2  West Germany       3           3       6         8
3     Argentina       2           3       5         5
4        France       2           1       3         6
5       Uruguay       2           0       2         5
6       Germany       1           1       2         5
7       England       1           0       1         3
8         Spain       1           0       1         2
9   Netherlands       0           3       3         5


**Interpretation.** Brazil is the most successful team, with 5 titles, 7 finals and 11 top-four finishes. Italy (4 titles) and West Germany (3) follow. Counting West Germany and Germany as one footballing nation would give 4 titles and 13 top-four finishes, so a broadcaster's 'most successful nations' graphic should state which convention it uses.

### Q02. All-time team records

**Business question:** What is each team's all-time World Cup record (played, won, drawn, lost, goals, win percentage), for teams with at least 10 matches?

**SQL concepts:** CTE, CASE, GROUP BY with HAVING, arithmetic on aggregates

**Expected result:** one row per team with >= 10 matches (49 rows), ordered by win percentage: team_name, played, won, drawn, lost, goals_for, goals_against, goal_diff, win_pct.

In [3]:
q02 = """
WITH results AS (
    SELECT ta.team_id,
           ta.goals_for,
           ta.goals_against,
           CASE WHEN ta.goals_for > ta.goals_against THEN 1 ELSE 0 END AS won,
           CASE WHEN ta.goals_for = ta.goals_against THEN 1 ELSE 0 END AS drawn,
           CASE WHEN ta.goals_for < ta.goals_against THEN 1 ELSE 0 END AS lost
    FROM team_appearances ta
)
SELECT t.team_name,
       COUNT(*)                                   AS played,
       SUM(r.won)                                 AS won,
       SUM(r.drawn)                               AS drawn,
       SUM(r.lost)                                AS lost,
       SUM(r.goals_for)                           AS goals_for,
       SUM(r.goals_against)                       AS goals_against,
       SUM(r.goals_for) - SUM(r.goals_against)    AS goal_diff,
       ROUND(100.0 * SUM(r.won) / COUNT(*), 1)    AS win_pct
FROM results r
JOIN teams t ON t.team_id = r.team_id
GROUP BY t.team_name
HAVING COUNT(*) >= 10
ORDER BY win_pct DESC, played DESC;
"""
df = pd.read_sql_query(q02, engine)
R["Q02"] = df
print(df.head(10))

      team_name  played  won  drawn  lost  goals_for  goals_against  goal_diff  win_pct
0        Brazil     109   73     18    18        229            105        124     67.0
1       Germany      47   31      6    10         95             48         47     66.0
2  West Germany      62   36     14    12        131             77         54     58.1
3         Italy      83   45     21    17        128             77         51     54.2
4   Netherlands      50   27     12    11         86             48         38     54.0
5     Argentina      81   43     15    23        137             93         44     53.1
6        France      66   34     13    19        120             77         43     51.5
7        Turkey      10    5      1     4         20             17          3     50.0
8  Soviet Union      31   15      6    10         53             34         19     48.4
9       Croatia      23   11      4     8         35             26          9     47.8


**Interpretation.** Among teams with at least 10 matches, Brazil has the best record (73 wins in 109 matches, 67.0%), followed by Germany and West Germany; Bulgaria won only 3 of 26. Because a shoot-out counts as a draw, these figures follow FIFA's convention. Analysts should keep the minimum-matches filter, since small samples such as Turkey's 10 matches inflate percentages.

### Q03. Host-country advantage

**Business question:** Do host nations do better than other teams? Compare win, draw and loss rates and average goal difference, overall and for two eras (1930-1978, 1982-2018).

**SQL concepts:** CTE, LEFT JOIN (to flag hosts), CASE, GROUP BY, UNION ALL, AVG of 0/1 values

**Expected result:** 6 rows: (host | other teams) x (1930-1978 | 1982-2018 | all years): side, era, matches, win_pct, draw_pct, loss_pct, avg_goal_diff.

In [4]:
q03 = """
WITH flagged AS (
    SELECT CASE WHEN h.team_id IS NOT NULL THEN 'host' ELSE 'other teams' END     AS side,
           CASE WHEN tr.year < 1982 THEN '1930-1978' ELSE '1982-2018' END           AS era,
           ta.goals_for - ta.goals_against                                          AS goal_diff,
           CASE WHEN ta.goals_for > ta.goals_against THEN 1.0 ELSE 0.0 END          AS won,
           CASE WHEN ta.goals_for = ta.goals_against THEN 1.0 ELSE 0.0 END          AS drawn,
           CASE WHEN ta.goals_for < ta.goals_against THEN 1.0 ELSE 0.0 END          AS lost
    FROM team_appearances ta
    JOIN tournaments tr ON tr.tournament_id = ta.tournament_id
    LEFT JOIN host_countries h
           ON h.tournament_id = ta.tournament_id AND h.team_id = ta.team_id
)
SELECT side, era, COUNT(*) AS matches,
       ROUND(100 * AVG(won), 1)   AS win_pct,
       ROUND(100 * AVG(drawn), 1) AS draw_pct,
       ROUND(100 * AVG(lost), 1)  AS loss_pct,
       ROUND(AVG(goal_diff), 2)   AS avg_goal_diff
FROM flagged
GROUP BY side, era
UNION ALL
SELECT side, 'all years', COUNT(*),
       ROUND(100 * AVG(won), 1), ROUND(100 * AVG(drawn), 1), ROUND(100 * AVG(lost), 1),
       ROUND(AVG(goal_diff), 2)
FROM flagged
GROUP BY side
ORDER BY side, era;
"""
df = pd.read_sql_query(q03, engine)
R["Q03"] = df
print(df.head(10))

          side        era  matches  win_pct  draw_pct  loss_pct  avg_goal_diff
0         host  1930-1978       57     71.9      10.5      17.5           1.30
1         host  1982-2018       61     54.1      26.2      19.7           0.56
2         host  all years      118     62.7      18.6      18.6           0.92
3  other teams  1930-1978      559     38.3      17.9      43.8          -0.13
4  other teams  1982-2018     1123     36.8      24.6      38.6          -0.03
5  other teams  all years     1682     37.3      22.4      40.4          -0.06


**Interpretation.** Hosts won 62.7% of their matches against 37.3% for all other teams, with an average goal difference of +0.92 per match. The advantage has shrunk, from 71.9% before 1982 to 54.1% since, so a federation bidding to host can expect a real but smaller boost today.

### Q04. Confederation performance by decade

**Business question:** How does each confederation perform, decade by decade (win percentage of its teams' matches)?

**SQL concepts:** 3-table JOIN, GROUP BY on an expression (decade), conditional aggregation into columns (pivot), AVG ignores NULLs

**Expected result:** one row per decade with a World Cup (8 rows: 1930s, then 1950s-2010s): decade, matches, then win_pct for UEFA, CONMEBOL, CONCACAF, CAF, AFC, OFC (NULL when the confederation had no team that decade). Caveat: confederation is each team's CURRENT one (e.g. Australia counts as AFC throughout).

In [5]:
q04 = """
WITH res AS (
    SELECT (tr.year / 10) * 10 AS decade,
           c.confederation_code AS conf,
           CASE WHEN ta.goals_for > ta.goals_against THEN 100.0 ELSE 0.0 END AS win
    FROM team_appearances ta
    JOIN tournaments tr    ON tr.tournament_id = ta.tournament_id
    JOIN teams t           ON t.team_id = ta.team_id
    JOIN confederations c  ON c.confederation_id = t.confederation_id
)
SELECT decade,
       COUNT(*) / 2 AS matches,
       ROUND(AVG(CASE WHEN conf = 'UEFA'     THEN win END), 1) AS uefa_win_pct,
       ROUND(AVG(CASE WHEN conf = 'CONMEBOL' THEN win END), 1) AS conmebol_win_pct,
       ROUND(AVG(CASE WHEN conf = 'CONCACAF' THEN win END), 1) AS concacaf_win_pct,
       ROUND(AVG(CASE WHEN conf = 'CAF'      THEN win END), 1) AS caf_win_pct,
       ROUND(AVG(CASE WHEN conf = 'AFC'      THEN win END), 1) AS afc_win_pct,
       ROUND(AVG(CASE WHEN conf = 'OFC'      THEN win END), 1) AS ofc_win_pct
FROM res
GROUP BY decade
ORDER BY decade;
"""
df = pd.read_sql_query(q04, engine)
R["Q04"] = df
print(df.head(10))

   decade  matches  uefa_win_pct  conmebol_win_pct  concacaf_win_pct  caf_win_pct  afc_win_pct  ofc_win_pct
0    1930       53          46.3              55.6              30.0          0.0          0.0          NaN
1    1950       83          41.0              52.8               9.1          NaN          0.0          NaN
2    1960       64          44.6              42.9              16.7          NaN         25.0          NaN
3    1970      108          42.1              45.5              15.4         11.1          0.0          NaN
4    1980      104          37.6              47.2              21.4         23.1          0.0          0.0
5    1990      168          44.0              40.7              24.0         23.5         12.0          NaN
6    2000      128          44.5              54.1              24.0         21.2         21.2          NaN
7    2010      192          47.4              53.2              24.3         19.2         19.0          0.0


**Interpretation.** CONMEBOL teams have the highest win rate in most decades (about 41-56%), while UEFA teams are steady at 38-47%. CAF and AFC teams went from no wins in the 1930s to about 19-21% in the 2000s and 2010s, which matters in debates on allocating World Cup places. The confederation shown is each team's current one.

### Q05. Teams that never got past the opening round

**Business question:** Which teams have never progressed beyond the opening round (stage 1) of any World Cup they played in?

**SQL concepts:** correlated NOT EXISTS subquery, JOIN, GROUP BY, MIN/MAX

**Expected result:** one row per such team (30 rows): team_name, appearances, matches, first_year, last_year; most appearances first. Stage order (stage_number) is used rather than the label 'group stage', so the knockout-only tournaments of 1934 and 1938 are handled correctly.

In [6]:
q05 = """
SELECT t.team_name,
       COUNT(DISTINCT q.tournament_id) AS appearances,
       (SELECT COUNT(*) FROM team_appearances x WHERE x.team_id = t.team_id) AS matches,
       MIN(tr.year) AS first_year,
       MAX(tr.year) AS last_year
FROM qualified_teams q
JOIN teams t        ON t.team_id = q.team_id
JOIN tournaments tr ON tr.tournament_id = q.tournament_id
WHERE NOT EXISTS (
        SELECT 1
        FROM team_appearances ta
        JOIN matches m ON m.match_id = ta.match_id
        WHERE ta.team_id = q.team_id
          AND m.stage_number > 1)
GROUP BY t.team_id, t.team_name
ORDER BY appearances DESC, matches DESC, t.team_name;
"""
df = pd.read_sql_query(q05, engine)
R["Q05"] = df
print(df.head(10))

      team_name  appearances  matches  first_year  last_year
0      Scotland            8       23        1954       1998
1          Iran            5       15        1978       2018
2       Tunisia            5       15        1978       2018
3      Honduras            3        9        1982       2014
4   Ivory Coast            3        9        2006       2014
5  South Africa            3        9        1998       2010
6         Egypt            3        7        1934       2018
7       Bolivia            3        6        1930       1994
8   El Salvador            2        6        1970       1982
9   New Zealand            2        6        1982       2010


**Interpretation.** 30 teams have never progressed beyond the opening round. Scotland is the most persistent case (8 tournaments, 23 matches), followed by Iran and Tunisia (5 each). For these federations, reaching the second round would be a historic first, which makes this a clear benchmark of progress.

### Q06. Top-four finishers that never won the title

**Business question:** Which teams have finished in the top four but never won the World Cup, and what was their best finish?

**SQL concepts:** set operation EXCEPT inside a CTE, JOIN, GROUP BY, STRING_AGG with ORDER BY

**Expected result:** one row per team (16 rows): team_name, best_finish, top_four_finishes, years; best finish first (runners-up: Netherlands, Czechoslovakia, Hungary, Sweden, Croatia).

In [7]:
q06 = """
WITH never_won AS (
    SELECT team_id FROM tournament_standings
    EXCEPT
    SELECT team_id FROM tournament_standings WHERE position = 1
)
SELECT t.team_name,
       MIN(s.position) AS best_finish,
       COUNT(*)        AS top_four_finishes,
       STRING_AGG(CAST(tr.year AS VARCHAR(4)) || ' (' || CAST(s.position AS VARCHAR(1)) || ')', ', '
                  ORDER BY tr.year) AS years_and_positions
FROM never_won n
JOIN tournament_standings s ON s.team_id = n.team_id
JOIN teams t                ON t.team_id = n.team_id
JOIN tournaments tr         ON tr.tournament_id = s.tournament_id
GROUP BY t.team_name
ORDER BY best_finish, top_four_finishes DESC, t.team_name;
"""
df = pd.read_sql_query(q06, engine)
R["Q06"] = df
print(df.head(10))

        team_name  best_finish  top_four_finishes                               years_and_positions
0     Netherlands            2                  5  1974 (2), 1978 (2), 1998 (4), 2010 (2), 2014 (3)
1          Sweden            2                  4            1938 (4), 1950 (3), 1958 (2), 1994 (3)
2         Croatia            2                  2                                1998 (3), 2018 (2)
3  Czechoslovakia            2                  2                                1934 (2), 1962 (2)
4         Hungary            2                  2                                1938 (2), 1954 (2)
5         Austria            3                  2                                1934 (4), 1954 (3)
6         Belgium            3                  2                                1986 (4), 2018 (3)
7          Poland            3                  2                                1974 (3), 1982 (3)
8        Portugal            3                  2                                1966 (3), 2006 (4)


**Interpretation.** 16 teams have reached the top four without ever winning. The Netherlands stand out, with three lost finals (1974, 1978, 2010) and five top-four finishes; Sweden, Croatia, Czechoslovakia and Hungary have also lost finals.

### Q07. Head-to-head record between two teams (example: Brazil v Sweden)

**Business question:** What is the complete World Cup head-to-head record between two chosen teams? (Change the two names in the params CTE to compare any pair.)

**SQL concepts:** SELF-JOIN of team_appearances (a team's row paired with its opponent's row in the same match), parameter CTE, CASE, running totals with window SUM() OVER

**Expected result:** one row per meeting in date order (7 rows for Brazil v Sweden): year, stage, score from team A's view, result, and running wins/draws for each side.

In [8]:
q07 = """
WITH params AS (
    SELECT 'Brazil' AS team_a, 'Sweden' AS team_b
),
h2h AS (
    SELECT m.match_date, tr.year, st.stage_name,
           a.goals_for AS a_goals, b.goals_for AS b_goals,
           CASE WHEN a.goals_for > b.goals_for THEN 1 ELSE 0 END AS a_win,
           CASE WHEN a.goals_for = b.goals_for THEN 1 ELSE 0 END AS draw,
           CASE WHEN a.goals_for < b.goals_for THEN 1 ELSE 0 END AS b_win
    FROM team_appearances a
    JOIN team_appearances b      ON b.match_id = a.match_id AND b.team_id <> a.team_id   -- self-join
    JOIN teams ta                ON ta.team_id = a.team_id
    JOIN teams tb                ON tb.team_id = b.team_id
    JOIN params p                ON ta.team_name = p.team_a AND tb.team_name = p.team_b
    JOIN matches m               ON m.match_id = a.match_id
    JOIN tournaments tr          ON tr.tournament_id = m.tournament_id
    JOIN tournament_stages st    ON st.tournament_id = m.tournament_id AND st.stage_number = m.stage_number
)
SELECT year, stage_name,
       CAST(a_goals AS VARCHAR(2)) || '-' || CAST(b_goals AS VARCHAR(2)) AS score_a_b,
       CASE WHEN a_win = 1 THEN 'team A win' WHEN draw = 1 THEN 'draw' ELSE 'team B win' END AS result,
       SUM(a_win) OVER (ORDER BY match_date) AS a_wins_so_far,
       SUM(draw)  OVER (ORDER BY match_date) AS draws_so_far,
       SUM(b_win) OVER (ORDER BY match_date) AS b_wins_so_far
FROM h2h
ORDER BY match_date;
"""
df = pd.read_sql_query(q07, engine)
R["Q07"] = df
print(df.head(10))

   year         stage_name score_a_b      result  a_wins_so_far  draws_so_far  b_wins_so_far
0  1938  third-place match       4-2  team A win              1             0              0
1  1950        final round       7-1  team A win              2             0              0
2  1958              final       5-2  team A win              3             0              0
3  1978        group stage       1-1        draw              3             1              0
4  1990        group stage       2-1  team A win              4             1              0
5  1994        group stage       1-1        draw              4             2              0
6  1994        semi-finals       1-0  team A win              5             2              0


**Interpretation.** Brazil are unbeaten against Sweden in seven World Cup meetings (5 wins, 2 draws), including the 1958 final and the 1994 semi-final. Changing the two names in `params` produces the same history for any pair, which is useful to coaches and broadcasters before a match.

### Q08. Cumulative titles over time

**Business question:** How did each country's title count build up over time, and who led the all-time table after each tournament?

**SQL concepts:** window functions - running total COUNT(*) OVER (PARTITION BY ... ORDER BY ...), correlated subquery for the leader

**Expected result:** one row per tournament (21 rows): year, champion, champion_titles_so_far, most_titles_after (the team(s) with most titles after that tournament).

In [9]:
q08 = """
WITH champs AS (
    SELECT tr.year, s.team_id, t.team_name,
           COUNT(*) OVER (PARTITION BY s.team_id ORDER BY tr.year
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS titles_so_far
    FROM tournament_standings s
    JOIN tournaments tr ON tr.tournament_id = s.tournament_id
    JOIN teams t        ON t.team_id = s.team_id
    WHERE s.position = 1
)
SELECT c.year,
       c.team_name     AS champion,
       c.titles_so_far AS champion_titles_so_far,
       (SELECT STRING_AGG(x.team_name || ' ' || CAST(x.n AS VARCHAR(2)), ', ' ORDER BY x.team_name)
        FROM (SELECT team_name, COUNT(*) AS n FROM champs c2 WHERE c2.year <= c.year GROUP BY team_name) x
        WHERE x.n = (SELECT MAX(n2) FROM (SELECT COUNT(*) AS n2 FROM champs c3
                                          WHERE c3.year <= c.year GROUP BY team_name) y)
       ) AS most_titles_after
FROM champs c
ORDER BY c.year;
"""
df = pd.read_sql_query(q08, engine)
R["Q08"] = df
print(df.head(10))

   year      champion  champion_titles_so_far             most_titles_after
0  1930       Uruguay                       1                     Uruguay 1
1  1934         Italy                       1            Italy 1, Uruguay 1
2  1938         Italy                       2                       Italy 2
3  1950       Uruguay                       2            Italy 2, Uruguay 2
4  1954  West Germany                       1            Italy 2, Uruguay 2
5  1958        Brazil                       1            Italy 2, Uruguay 2
6  1962        Brazil                       2  Brazil 2, Italy 2, Uruguay 2
7  1966       England                       1  Brazil 2, Italy 2, Uruguay 2
8  1970        Brazil                       3                      Brazil 3
9  1974  West Germany                       2                      Brazil 3


**Interpretation.** Uruguay, Italy and Brazil traded the lead in the early years; after 1962 they were tied on two titles each. Brazil took the sole lead in 1970, were caught by Italy in 1982 and by Italy and West Germany in 1990, and have led alone since their fourth title in 1994.

### Q09. Comebacks: winning after conceding the first goal

**Business question:** Which teams most often won a match after conceding the first goal, and how often do they manage it?

**SQL concepts:** CTE, window function ROW_NUMBER() OVER (PARTITION BY match ORDER BY minute) to find each match's first goal, JOIN, conditional aggregation, HAVING

**Expected result:** one row per team that conceded first at least 5 times (55 rows): team_name, conceded_first, comeback_wins, comeback_draws, comeback_win_pct; most comeback wins first.

In [10]:
q09 = """
WITH ordered_goals AS (
    SELECT g.match_id, g.team_id,
           ROW_NUMBER() OVER (PARTITION BY g.match_id
                              ORDER BY g.minute_regulation, g.minute_stoppage, g.goal_id) AS rn
    FROM goals g
),
first_goal AS (
    SELECT match_id, team_id AS first_scoring_team FROM ordered_goals WHERE rn = 1
)
SELECT t.team_name,
       COUNT(*) AS conceded_first,
       SUM(CASE WHEN ta.goals_for > ta.goals_against THEN 1 ELSE 0 END) AS comeback_wins,
       SUM(CASE WHEN ta.goals_for = ta.goals_against THEN 1 ELSE 0 END) AS comeback_draws,
       ROUND(100.0 * SUM(CASE WHEN ta.goals_for > ta.goals_against THEN 1 ELSE 0 END) / COUNT(*), 1)
           AS comeback_win_pct
FROM team_appearances ta
JOIN first_goal f ON f.match_id = ta.match_id AND f.first_scoring_team <> ta.team_id
JOIN teams t      ON t.team_id = ta.team_id
GROUP BY t.team_name
HAVING COUNT(*) >= 5
ORDER BY comeback_wins DESC, comeback_win_pct DESC, t.team_name;
"""
df = pd.read_sql_query(q09, engine)
R["Q09"] = df
print(df.head(10))

      team_name  conceded_first  comeback_wins  comeback_draws  comeback_win_pct
0        Brazil              33             15               3              45.5
1  West Germany              22             10               3              45.5
2        Sweden              19              6               2              31.6
3         Spain              24              6               5              25.0
4   Netherlands              16              5               2              31.3
5         Italy              23              5               6              21.7
6       Uruguay              25              5               3              20.0
7        France              22              4               4              18.2
8     Argentina              29              4               4              13.8
9       Austria              19              3               3              15.8


**Interpretation.** Brazil (15) and West Germany (10) most often won after conceding the first goal, each turning 45.5% of such matches into wins. Most teams rarely recover; Argentina won only 4 of 29. Scoring first is therefore a strong predictor of the result, a point coaches and in-game analysts can use.

## 2. Player performance

### Q10. All-time top 10 goal scorers (own goals excluded)

**Business question:** Who are the World Cup's all-time top 10 goal scorers?

**SQL concepts:** JOIN, WHERE filter, GROUP BY, window function RANK() in a derived table, filtering on the rank (so players tied for a place are all kept), STRING_AGG(DISTINCT ...)

**Expected result:** every player ranked 10th or better, ties included: rank, player, team, goals, penalties, tournaments, matches_scored_in. Miroslav Klose first with 16.

In [11]:
q10 = """
SELECT *
FROM (
SELECT RANK() OVER (ORDER BY COUNT(*) DESC)                  AS rank,
       COALESCE(p.given_name || ' ', '') || p.family_name    AS player,
       STRING_AGG(DISTINCT t.team_name, ' / ')               AS team,
       COUNT(*)                                              AS goals,
       SUM(CASE WHEN g.penalty THEN 1 ELSE 0 END)            AS penalties,
       COUNT(DISTINCT g.tournament_id)                       AS tournaments,
       COUNT(DISTINCT g.match_id)                            AS matches_scored_in
FROM goals g
JOIN players p ON p.player_id = g.player_id
JOIN teams t   ON t.team_id = g.player_team_id
WHERE NOT g.own_goal
GROUP BY p.player_id, p.given_name, p.family_name
) ranked
WHERE rank <= 10
ORDER BY rank, player;
"""
df = pd.read_sql_query(q10, engine)
R["Q10"] = df
print(df.head(10))

   rank             player                    team  goals  penalties  tournaments  matches_scored_in
0     1     Miroslav Klose                 Germany     16          0            4                 11
1     2            Ronaldo                  Brazil     15          1            3                 11
2     3        Gerd Müller            West Germany     14          1            2                  9
3     4      Just Fontaine                  France     13          0            1                  6
4     5               Pelé                  Brazil     12          0            4                  8
5     6   Jürgen Klinsmann  Germany / West Germany     11          0            3                 10
6     6      Sándor Kocsis                 Hungary     11          0            1                  4
7     8  Gabriel Batistuta               Argentina     10          4            3                  6
8     8       Gary Lineker                 England     10          2            2          

**Interpretation.** Miroslav Klose is the all-time top scorer with 16 goals over four tournaments, one ahead of Ronaldo. Just Fontaine's 13 goals all came in one tournament (1958), and six players share eighth place with 10 goals (13 rows in total because ties are kept); Gabriel Batistuta scored 4 of his 10 from penalties.

### Q11. Top scorer of each tournament vs the official Golden Boot

**Business question:** Who scored the most goals at each tournament, and does that agree with the official Golden Boot award?

**SQL concepts:** CTEs, window function RANK() OVER (PARTITION BY tournament ORDER BY goals DESC), correlated scalar subqueries, set comparison with EXCEPT inside NOT EXISTS

**Expected result:** one row per tournament (21 rows, newest first): year, top_scorers (ties listed), goals, golden_boot (official winner(s)), agreement ('same' / 'different').

In [12]:
q11 = """
WITH tally AS (
    SELECT tournament_id, player_id, COUNT(*) AS goals
    FROM goals
    WHERE NOT own_goal
    GROUP BY tournament_id, player_id
),
top_scorers AS (
    SELECT tournament_id, player_id, goals
    FROM (SELECT tally.*, RANK() OVER (PARTITION BY tournament_id ORDER BY goals DESC) AS rnk
          FROM tally) r
    WHERE rnk = 1
),
golden_boot AS (
    SELECT aw.tournament_id, aw.player_id
    FROM award_winners aw
    JOIN awards a ON a.award_id = aw.award_id
    WHERE a.award_name = 'Golden Boot'
)
SELECT tr.year,
       (SELECT STRING_AGG(COALESCE(p.given_name || ' ', '') || p.family_name, ', '
                          ORDER BY p.family_name)
        FROM top_scorers ts JOIN players p ON p.player_id = ts.player_id
        WHERE ts.tournament_id = tr.tournament_id)                        AS top_scorers,
       (SELECT MAX(goals) FROM top_scorers ts WHERE ts.tournament_id = tr.tournament_id) AS goals,
       (SELECT STRING_AGG(COALESCE(p.given_name || ' ', '') || p.family_name, ', '
                          ORDER BY p.family_name)
        FROM golden_boot gb JOIN players p ON p.player_id = gb.player_id
        WHERE gb.tournament_id = tr.tournament_id)                        AS golden_boot,
       CASE WHEN NOT EXISTS (SELECT player_id FROM top_scorers WHERE tournament_id = tr.tournament_id
                             EXCEPT
                             SELECT player_id FROM golden_boot WHERE tournament_id = tr.tournament_id)
             AND NOT EXISTS (SELECT player_id FROM golden_boot WHERE tournament_id = tr.tournament_id
                             EXCEPT
                             SELECT player_id FROM top_scorers WHERE tournament_id = tr.tournament_id)
            THEN 'same' ELSE 'different' END                               AS agreement
FROM tournaments tr
ORDER BY tr.year DESC;
"""
df = pd.read_sql_query(q11, engine)
R["Q11"] = df
print(df.head(10))

   year                                                top_scorers  goals                     golden_boot  agreement
0  2018                                                 Harry Kane      6                      Harry Kane       same
1  2014                                            James Rodríguez      6                 James Rodríguez       same
2  2010  Diego Forlán, Thomas Müller, Wesley Sneijder, David Villa      5                   Thomas Müller  different
3  2006                                             Miroslav Klose      5                  Miroslav Klose       same
4  2002                                                    Ronaldo      8                         Ronaldo       same
5  1998                                                Davor Šuker      6                     Davor Šuker       same
6  1994                             Oleg Salenko, Hristo Stoichkov      6  Oleg Salenko, Hristo Stoichkov       same
7  1990                                        Salvatore Schilla

**Interpretation.** In 20 of 21 tournaments the official Golden Boot went to the player or players with the most goals; earlier ties were simply shared (six players in 1962). The exception is 2010: four players scored 5 goals, but the award went to Thomas Müller alone. FIFA broke that tie on a further criterion (assists) that the goals table alone cannot reproduce.

### Q12. Hat-tricks

**Business question:** Which players scored three or more goals in a single match, and who holds the record for most goals in one match?

**SQL concepts:** multi-table JOIN, GROUP BY, HAVING COUNT(*) >= 3

**Expected result:** one row per hat-trick (52 rows), most goals first: year, player, team, opponent, final score, stage, goals. Oleg Salenko's 5 goals (1994) first.

In [13]:
q12 = """
SELECT tr.year,
       COALESCE(p.given_name || ' ', '') || p.family_name               AS player,
       t.team_name                                                      AS team,
       o.team_name                                                      AS opponent,
       CAST(ta.goals_for AS VARCHAR(2)) || '-' || CAST(ta.goals_against AS VARCHAR(2)) AS final_score,
       st.stage_name,
       COUNT(*)                                                         AS goals
FROM goals g
JOIN players p             ON p.player_id = g.player_id
JOIN team_appearances ta   ON ta.match_id = g.match_id AND ta.team_id = g.player_team_id
JOIN teams t               ON t.team_id = ta.team_id
JOIN teams o               ON o.team_id = ta.opponent_id
JOIN matches m             ON m.match_id = g.match_id
JOIN tournament_stages st  ON st.tournament_id = m.tournament_id AND st.stage_number = m.stage_number
JOIN tournaments tr        ON tr.tournament_id = m.tournament_id
WHERE NOT g.own_goal
GROUP BY tr.year, m.match_id, p.player_id, p.given_name, p.family_name, t.team_name, o.team_name,
         ta.goals_for, ta.goals_against, st.stage_name
HAVING COUNT(*) >= 3
ORDER BY goals DESC, tr.year, player;
"""
df = pd.read_sql_query(q12, engine)
R["Q12"] = df
print(df.head(10))

   year             player           team      opponent final_score         stage_name  goals
0  1994       Oleg Salenko         Russia      Cameroon         6-1        group stage      5
1  1938   Ernst Wilimowski         Poland        Brazil         5-6        round of 16      4
2  1950             Ademir         Brazil        Sweden         7-1        final round      4
3  1954      Sándor Kocsis        Hungary  West Germany         8-3        group stage      4
4  1958      Just Fontaine         France  West Germany         6-3  third-place match      4
5  1966            Eusébio       Portugal   North Korea         5-3     quarter-finals      4
6  1986  Emilio Butragueño          Spain       Denmark         5-1        round of 16      4
7  1930     Bert Patenaude  United States      Paraguay         3-0        group stage      3
8  1930  Guillermo Stábile      Argentina        Mexico         6-3        group stage      3
9  1930          Pedro Cea        Uruguay    Yugoslavia     

**Interpretation.** There have been 52 hat-tricks. Oleg Salenko's five goals against Cameroon (1994) are the record for one match, and six players have scored four; Ernst Wilimowski scored four for Poland in 1938 and still lost 5-6 to Brazil.

### Q13. Goals scored by substitutes (1970 onwards)

**Business question:** How many goals were scored by substitutes at each tournament since 1970, and what share of all goals is that?

**SQL concepts:** JOIN on a composite key (match_id, player_id), CASE, GROUP BY, ratio

**Expected result:** one row per tournament 1970-2018 (13 rows, newest first): year, goals, substitute_goals, substitute_pct. Line-ups exist only from 1970, so earlier years cannot appear.

In [14]:
q13 = """
SELECT tr.year,
       COUNT(*)                                                       AS goals,
       SUM(CASE WHEN NOT pa.starter THEN 1 ELSE 0 END)                AS substitute_goals,
       ROUND(100.0 * SUM(CASE WHEN NOT pa.starter THEN 1 ELSE 0 END) / COUNT(*), 1) AS substitute_pct
FROM goals g
JOIN player_appearances pa ON pa.match_id = g.match_id AND pa.player_id = g.player_id
JOIN tournaments tr        ON tr.tournament_id = g.tournament_id
WHERE NOT g.own_goal
GROUP BY tr.year
ORDER BY tr.year DESC;
"""
df = pd.read_sql_query(q13, engine)
R["Q13"] = df
print(df.head(10))

   year  goals  substitute_goals  substitute_pct
0  2018    157                15             9.6
1  2014    166                32            19.3
2  2010    143                15            10.5
3  2006    143                23            16.1
4  2002    158                21            13.3
5  1998    165                14             8.5
6  1994    140                13             9.3
7  1990    115                20            17.4
8  1986    130                11             8.5
9  1982    145                16            11.0

**Interpretation.** Substitutes scored about 5% of goals in the 1970s, but between 9.6% and 19.3% in every tournament since 2006, peaking at 32 goals (19.3%) in 2014. Benches have become a real attacking weapon, which coaches can plan for.

### Q14. Average squad age by position and tournament

**Business question:** How old are World Cup squads, by position, and how has that changed?

**SQL concepts:** date arithmetic (date - date = days), conditional aggregation pivot, AVG ignoring NULLs, COUNT(*) vs COUNT(column) to report missing birth dates

**Expected result:** one row per tournament (21 rows, newest first): year, avg age of GK, DF, MF, FW and all players at the tournament's start date, and players_without_birth_date.

In [15]:
q14 = """
WITH ages AS (
    SELECT tr.year, s.position_code,
           (tr.start_date - p.birth_date) / 365.25 AS age     -- NULL when the birth date is unknown
    FROM squads s
    JOIN players p      ON p.player_id = s.player_id
    JOIN tournaments tr ON tr.tournament_id = s.tournament_id
)
SELECT year,
       ROUND(AVG(CASE WHEN position_code = 'GK' THEN age END), 1) AS gk_avg_age,
       ROUND(AVG(CASE WHEN position_code = 'DF' THEN age END), 1) AS df_avg_age,
       ROUND(AVG(CASE WHEN position_code = 'MF' THEN age END), 1) AS mf_avg_age,
       ROUND(AVG(CASE WHEN position_code = 'FW' THEN age END), 1) AS fw_avg_age,
       ROUND(AVG(age), 1)                                         AS all_avg_age,
       COUNT(*) - COUNT(age)                                      AS players_without_birth_date
FROM ages
GROUP BY year
ORDER BY year DESC;
"""
df = pd.read_sql_query(q14, engine)
R["Q14"] = df
print(df.head(10))

   year  gk_avg_age  df_avg_age  mf_avg_age  fw_avg_age  all_avg_age  players_without_birth_date
0  2018        29.6        28.1        27.6        27.0         27.9                           0
1  2014        28.9        27.5        26.8        27.0         27.3                           0
2  2010        29.0        27.5        26.7        27.1         27.3                           0
3  2006        29.1        27.5        26.9        26.8         27.4                           0
4  2002        29.2        27.5        27.2        26.7         27.4                           0
5  1998        29.3        27.6        27.1        26.8         27.5                           0
6  1994        28.8        27.5        26.9        27.0         27.4                           0
7  1990        28.8        27.1        26.2        26.3         26.9                           0
8  1986        28.8        27.3        26.5        26.5         27.1                           0
9  1982        28.5        27.

**Interpretation.** Squads have got older: average age rose from 24.9 in 1930 to 27.9 in 2018. Goalkeepers have been the oldest group at every tournament since 1958 (29.6 in 2018), and forwards and midfielders are usually the youngest. Birth dates are missing for 79 squad members, all from 1962 or earlier, and they are left out of the averages.

### Q15. Most World Cup appearances for each nation (1970 onwards)

**Business question:** Who has made the most World Cup appearances for each national team?

**SQL concepts:** CTE, window function ROW_NUMBER() OVER (PARTITION BY team ORDER BY ...), filter on the row number, JOIN

**Expected result:** one row per team that played from 1970 on (81 rows), most appearances first: team_name, player, appearances, tournaments, as_starter. Miroslav Klose first (24 for Germany). Ties are broken by tournaments, then player id. Note: the overall record holder Lothar Matthaeus (25) does not appear, because his matches are split between West Germany and Germany.

In [16]:
q15 = """
WITH apps AS (
    SELECT team_id, player_id,
           COUNT(*)                                        AS appearances,
           COUNT(DISTINCT tournament_id)                   AS tournaments,
           SUM(CASE WHEN starter THEN 1 ELSE 0 END)        AS as_starter
    FROM player_appearances
    GROUP BY team_id, player_id
),
ranked AS (
    SELECT apps.*,
           ROW_NUMBER() OVER (PARTITION BY team_id
                              ORDER BY appearances DESC, tournaments DESC, player_id) AS rn
    FROM apps
)
SELECT t.team_name,
       COALESCE(p.given_name || ' ', '') || p.family_name AS player,
       r.appearances, r.tournaments, r.as_starter
FROM ranked r
JOIN teams t   ON t.team_id = r.team_id
JOIN players p ON p.player_id = r.player_id
WHERE r.rn = 1
ORDER BY r.appearances DESC, t.team_name;
"""
df = pd.read_sql_query(q15, engine)
R["Q15"] = df
print(df.head(10))

      team_name           player  appearances  tournaments  as_starter
0       Germany   Miroslav Klose           24            4          22
1         Italy    Paolo Maldini           23            4          23
2     Argentina   Diego Maradona           21            4          21
3        Poland  Władysław Żmuda           21            4          20
4        Brazil             Cafu           20            4          17
5        Mexico   Rafael Márquez           19            5          17
6  West Germany      Berti Vogts           19            3          19
7       Belgium       Enzo Scifo           17            4          17
8       England    Peter Shilton           17            3          17
9        France    Thierry Henry           17            4          12


**Interpretation.** Miroslav Klose (24 for Germany) and Paolo Maldini (23 for Italy) have the most appearances for a single team. Lothar Matthäus, the overall record holder with 25, tops neither list because his appearances are split between West Germany and Germany (see validation check V12). This is a direct result of counting historical teams separately.

### Q16. Players who represented more than one national team

**Business question:** Which players were in World Cup squads for more than one national team?

**SQL concepts:** GROUP BY, HAVING COUNT(DISTINCT ...) > 1, STRING_AGG with ORDER BY

**Expected result:** one row per player (34 rows): player, teams, squads (team (year) list). Most are "renamed state" cases (West Germany -> Germany, Yugoslavia -> Croatia, ...); a few are genuine switches of nationality from the 1930s-1960s.

In [17]:
q16 = """
SELECT COALESCE(p.given_name || ' ', '') || p.family_name                 AS player,
       COUNT(DISTINCT s.team_id)                                          AS teams,
       STRING_AGG(t.team_name || ' (' || CAST(tr.year AS VARCHAR(4)) || ')', ', ' ORDER BY tr.year) AS squads
FROM squads s
JOIN players p      ON p.player_id = s.player_id
JOIN teams t        ON t.team_id = s.team_id
JOIN tournaments tr ON tr.tournament_id = s.tournament_id
GROUP BY p.player_id, p.given_name, p.family_name
HAVING COUNT(DISTINCT s.team_id) > 1
ORDER BY teams DESC, MIN(tr.year), player;
"""
df = pd.read_sql_query(q16, engine)
R["Q16"] = df
print(df.head(10))

              player  teams                                                       squads
0    Dejan Stanković      3  Yugoslavia (1998), Serbia and Montenegro (2006), Serbia ...
1    Attilio Demaría      2                               Argentina (1930), Italy (1934)
2         Luis Monti      2                               Argentina (1930), Italy (1934)
3       Franz Wagner      2                               Austria (1934), Germany (1938)
4        Josef Stroh      2                               Austria (1934), Germany (1938)
5       Rudolf Raftl      2                               Austria (1934), Germany (1938)
6  Willibald Schmaus      2                               Austria (1934), Germany (1938)
7      Ferenc Puskás      2                                 Hungary (1954), Spain (1962)
8    José Santamaría      2                                 Uruguay (1954), Spain (1962)
9      José Altafini      2                                  Brazil (1958), Italy (1962)

**Interpretation.** 34 players appeared for more than one national team. Most reflect states that were renamed or split: West Germany to Germany, and Yugoslavia to Serbia and Montenegro to Serbia (Dejan Stanković played for all three). A few are genuine changes of nationality in the 1930s-1960s, such as Luis Monti (Argentina 1930, Italy 1934) and Ferenc Puskás (Hungary 1954, Spain 1962).

## 3. Goals and scoring patterns

### Q17. Goals per match by tournament, with change from the previous edition

**Business question:** How many goals per match were scored at each tournament, and how did that change from the previous World Cup?

**SQL concepts:** CTE, GROUP BY, window function LAG() OVER (ORDER BY year)

**Expected result:** one row per tournament (21 rows, oldest first): year, matches, goals, goals_per_match, previous_goals_per_match, change (NULL for 1930). Peak 5.38 in 1954.

In [18]:
q17 = """
WITH per_tournament AS (
    SELECT tr.year,
           COUNT(*)                                       AS matches,
           SUM(m.home_team_score + m.away_team_score)     AS goals,
           ROUND(1.0 * SUM(m.home_team_score + m.away_team_score) / COUNT(*), 2) AS goals_per_match
    FROM matches m
    JOIN tournaments tr ON tr.tournament_id = m.tournament_id
    GROUP BY tr.year
)
SELECT year, matches, goals, goals_per_match,
       LAG(goals_per_match) OVER (ORDER BY year)                    AS previous_goals_per_match,
       goals_per_match - LAG(goals_per_match) OVER (ORDER BY year)  AS change
FROM per_tournament
ORDER BY year;
"""
df = pd.read_sql_query(q17, engine)
R["Q17"] = df
print(df.head(10))

   year  matches  goals  goals_per_match  previous_goals_per_match  change
0  1930       18     70             3.89                       NaN     NaN
1  1934       17     70             4.12                      3.89    0.23
2  1938       18     84             4.67                      4.12    0.55
3  1950       22     88             4.00                      4.67   -0.67
4  1954       26    140             5.38                      4.00    1.38
5  1958       35    126             3.60                      5.38   -1.78
6  1962       32     89             2.78                      3.60   -0.82
7  1966       32     89             2.78                      2.78    0.00
8  1970       32     95             2.97                      2.78    0.19
9  1974       38     97             2.55                      2.97   -0.42


**Interpretation.** Scoring peaked at 5.38 goals per match in 1954 and has stayed below 3 since 1962. The low point was 1990 (2.21); the next tournament rose by 0.50 goals per match, the largest increase since 1954, and recent tournaments sit at about 2.6-2.7.

### Q18. Biggest winning margins and highest-scoring matches

**Business question:** Which matches had the biggest winning margins, and which had the most goals?

**SQL concepts:** derived table, window function DENSE_RANK() (two different rankings), filtering on window results in an outer query, ABS, string concatenation

**Expected result:** every match ranked in the top 2 for margin OR total goals (9 rows): margin_rank, goals_rank, year, stage, match (home score-score away), margin, total_goals.

In [19]:
q18 = """
SELECT margin_rank, goals_rank, year, stage_name, match, margin, total_goals
FROM (
    SELECT DENSE_RANK() OVER (ORDER BY ABS(m.home_team_score - m.away_team_score) DESC) AS margin_rank,
           DENSE_RANK() OVER (ORDER BY m.home_team_score + m.away_team_score DESC)       AS goals_rank,
           tr.year, st.stage_name,
           h.team_name || ' ' || CAST(m.home_team_score AS VARCHAR(2)) || '-'
               || CAST(m.away_team_score AS VARCHAR(2)) || ' ' || a.team_name             AS match,
           ABS(m.home_team_score - m.away_team_score)                                    AS margin,
           m.home_team_score + m.away_team_score                                         AS total_goals
    FROM matches m
    JOIN teams h               ON h.team_id = m.home_team_id
    JOIN teams a               ON a.team_id = m.away_team_id
    JOIN tournaments tr        ON tr.tournament_id = m.tournament_id
    JOIN tournament_stages st  ON st.tournament_id = m.tournament_id AND st.stage_number = m.stage_number
) ranked
WHERE margin_rank <= 2 OR goals_rank <= 2
ORDER BY margin DESC, total_goals DESC, year;
"""
df = pd.read_sql_query(q18, engine)
R["Q18"] = df
print(df.head(10))

   margin_rank  goals_rank  year      stage_name                     match  margin  total_goals
0            1           2  1982     group stage  Hungary 10-1 El Salvador       9           11
1            1           4  1954     group stage   Hungary 9-0 South Korea       9            9
2            1           4  1974     group stage      Yugoslavia 9-0 Zaire       9            9
3            2           5  1938  quarter-finals           Sweden 8-0 Cuba       8            8
4            2           5  1950     group stage       Uruguay 8-0 Bolivia       8            8
5            2           5  2002     group stage  Germany 8-0 Saudi Arabia       8            8
6            5           2  1954     group stage  Hungary 8-3 West Germany       5           11
7            8           1  1954  quarter-finals   Austria 7-5 Switzerland       2           12
8            9           2  1938     round of 16         Brazil 6-5 Poland       1           11


**Interpretation.** Three matches share the record winning margin of nine goals; the most emphatic was Hungary's 10-1 win over El Salvador in 1982. The highest-scoring match is Austria 7-5 Switzerland, the 1954 quarter-final (12 goals), and the 1954 tournament supplies several entries in this list.

### Q19. When are goals scored?

**Business question:** In which periods of a match are goals scored (15-minute bands, stoppage time, extra time)?

**SQL concepts:** CASE bucketing, GROUP BY, window function SUM() OVER () for the share of the total

**Expected result:** 10 rows in match order: period, goals, pct_of_all_goals, own_goals, penalties. Stoppage-time goals are listed separately from the band they follow.

In [20]:
q19 = """
WITH banded AS (
    SELECT CASE
             WHEN minute_regulation <= 45 AND minute_stoppage > 0 THEN '04 first-half stoppage time'
             WHEN minute_regulation <= 15                         THEN '01 minutes 1-15'
             WHEN minute_regulation <= 30                         THEN '02 minutes 16-30'
             WHEN minute_regulation <= 45                         THEN '03 minutes 31-45'
             WHEN minute_regulation <= 90 AND minute_stoppage > 0 THEN '08 second-half stoppage time'
             WHEN minute_regulation <= 60                         THEN '05 minutes 46-60'
             WHEN minute_regulation <= 75                         THEN '06 minutes 61-75'
             WHEN minute_regulation <= 90                         THEN '07 minutes 76-90'
             WHEN minute_regulation <= 105                        THEN '09 extra time, first half'
             ELSE                                                      '10 extra time, second half'
           END AS period,
           own_goal, penalty
    FROM goals
)
SELECT period,
       COUNT(*)                                                    AS goals,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)          AS pct_of_all_goals,
       SUM(CASE WHEN own_goal THEN 1 ELSE 0 END)                   AS own_goals,
       SUM(CASE WHEN penalty THEN 1 ELSE 0 END)                    AS penalties
FROM banded
GROUP BY period
ORDER BY period;
"""
df = pd.read_sql_query(q19, engine)
R["Q19"] = df
print(df.head(10))

                         period  goals  pct_of_all_goals  own_goals  penalties
0               01 minutes 1-15    334              13.1          8         22
1              02 minutes 16-30    373              14.6         10         36
2              03 minutes 31-45    344              13.5          8         34
3   04 first-half stoppage time     27               1.1          1          6
4              05 minutes 46-60    414              16.2          9         23
5              06 minutes 61-75    452              17.7          7         24
6              07 minutes 76-90    473              18.6          5         38
7  08 second-half stoppage time     64               2.5          3          9
8     09 extra time, first half     33               1.3          1          2
9    10 extra time, second half     34               1.3          0          3


**Interpretation.** Goals become more frequent as a match goes on: 13.1% come in the first 15 minutes, but 18.6% come in minutes 76-90, plus another 2.5% in second-half stoppage time. Coaches can use this to plan fitness and late substitutions, and broadcasters should expect decisive moments late.

### Q20. Penalty goals and own goals over time (VAR in 2018)

**Business question:** How have the shares of goals from in-match penalties and own goals changed over time, and was 2018 (the first World Cup with VAR) different?

**SQL concepts:** conditional aggregation SUM(CASE ...), GROUP BY, CASE labelling, window AVG() OVER with a frame to compare with the average of all earlier tournaments

**Expected result:** one row per tournament (21 rows, newest first): year, goals, penalty_goals, penalty_pct, own_goals, own_goal_pct, avg_penalty_pct_before (mean of all earlier editions), era.

In [21]:
q20 = """
WITH per AS (
    SELECT tr.year,
           COUNT(*)                                          AS goals,
           SUM(CASE WHEN g.penalty THEN 1 ELSE 0 END)        AS penalty_goals,
           SUM(CASE WHEN g.own_goal THEN 1 ELSE 0 END)       AS own_goals
    FROM goals g
    JOIN tournaments tr ON tr.tournament_id = g.tournament_id
    GROUP BY tr.year
)
SELECT year, goals,
       penalty_goals, ROUND(100.0 * penalty_goals / goals, 1) AS penalty_pct,
       own_goals,     ROUND(100.0 * own_goals / goals, 1)     AS own_goal_pct,
       ROUND(AVG(100.0 * penalty_goals / goals) OVER (ORDER BY year
             ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 1) AS avg_penalty_pct_before,
       CASE WHEN year >= 2018 THEN 'VAR' ELSE 'before VAR' END   AS era
FROM per
ORDER BY year DESC;
"""
df = pd.read_sql_query(q20, engine)
R["Q20"] = df
print(df.head(10))

   year  goals  penalty_goals  penalty_pct  own_goals  own_goal_pct  avg_penalty_pct_before         era
0  2018    169             22         13.0         12           7.1                     7.1         VAR
1  2014    171             12          7.0          5           2.9                     7.1  before VAR
2  2010    145              9          6.2          2           1.4                     7.1  before VAR
3  2006    147             13          8.8          4           2.7                     7.0  before VAR
4  2002    161             13          8.1          3           1.9                     6.9  before VAR
5  1998    171             17          9.9          6           3.5                     6.7  before VAR
6  1994    141             15         10.6          1           0.7                     6.5  before VAR
7  1990    115             13         11.3          0           0.0                     6.1  before VAR
8  1986    132             12          9.1          2           

**Interpretation.** 2018, the first World Cup with VAR, had the highest share of goals from penalties (22, 13.0%) and of own goals (12, 7.1%). The average penalty share of all earlier tournaments was 7.1%. Referees' bodies evaluating VAR would see a clear jump in penalties, although one tournament is a small sample.

### Q21. Penalty shoot-out records

**Business question:** Which teams win penalty shoot-outs, and how reliably do their players score their kicks?

**SQL concepts:** two CTEs (shoot-out results, kick statistics), JOIN, GROUP BY, HAVING

**Expected result:** one row per team with at least 2 shoot-outs (14 rows): team_name, shootouts, won, lost, kicks, scored, conversion_pct; most wins first. (West Germany and Germany are separate teams.)

In [22]:
q21 = """
WITH shootouts AS (
    SELECT team_id,
           COUNT(*)                                                           AS shootouts,
           SUM(CASE WHEN penalties_for > penalties_against THEN 1 ELSE 0 END) AS won
    FROM team_appearances
    WHERE penalties_for IS NOT NULL
    GROUP BY team_id
),
kicks AS (
    SELECT team_id,
           COUNT(*)                                     AS kicks,
           SUM(CASE WHEN converted THEN 1 ELSE 0 END)   AS scored
    FROM penalty_kicks
    GROUP BY team_id
)
SELECT t.team_name,
       s.shootouts, s.won, s.shootouts - s.won AS lost,
       k.kicks, k.scored,
       ROUND(100.0 * k.scored / k.kicks, 1)    AS conversion_pct
FROM shootouts s
JOIN kicks k ON k.team_id = s.team_id
JOIN teams t ON t.team_id = s.team_id
WHERE s.shootouts >= 2
ORDER BY s.won DESC, conversion_pct DESC, t.team_name;
"""
df = pd.read_sql_query(q21, engine)
R["Q21"] = df
print(df.head(10))

             team_name  shootouts  won  lost  kicks  scored  conversion_pct
0            Argentina          5    4     1     22      17            77.3
1         West Germany          3    3     0     14      13            92.9
2               Brazil          4    3     1     18      13            72.2
3               France          4    2     2     20      15            75.0
4              Croatia          2    2     0     10       7            70.0
5           Costa Rica          2    1     1     10       8            80.0
6               Russia          2    1     1      9       7            77.8
7  Republic of Ireland          2    1     1     10       7            70.0
8                Spain          4    1     3     19      13            68.4
9          Netherlands          3    1     2     12       8            66.7


**Interpretation.** Argentina have won the most shoot-outs (4 of 5), and West Germany won all three of theirs, scoring 13 of 14 kicks (92.9%). Among teams with four shoot-outs, England have the lowest conversion (57.9%, one win), and Mexico lost both of theirs, scoring only 2 of 7 kicks. This is useful material for coaches and sports psychologists preparing for shoot-outs.

## 4. Discipline and refereeing (1970 onwards)

### Q22. Strictest referees (1970 onwards)

**Business question:** Which referees show the most cards per match (minimum five matches), and how far above the overall average are they?

**SQL concepts:** CTE, LEFT JOIN (so card-free matches count as 0), GROUP BY, HAVING, scalar subquery for the overall average

**Expected result:** one row per referee with >= 5 matches since 1970 (34 rows), strictest first: referee, country, matches, cards, sendings_off, cards_per_match, above_average.

In [23]:
q22 = """
WITH ref_matches AS (
    SELECT ra.referee_id, ra.match_id,
           COUNT(b.booking_id)                                                    AS cards,
           SUM(CASE WHEN b.red_card OR b.second_yellow_card THEN 1 ELSE 0 END)    AS sendings_off
    FROM referee_appearances ra
    JOIN tournaments tr  ON tr.tournament_id = ra.tournament_id
    LEFT JOIN bookings b ON b.match_id = ra.match_id
    WHERE tr.year >= 1970
    GROUP BY ra.referee_id, ra.match_id
)
SELECT COALESCE(r.given_name || ' ', '') || r.family_name               AS referee,
       r.country_name                                                   AS country,
       COUNT(*)                                                         AS matches,
       CAST(SUM(rm.cards) AS INTEGER)                                   AS cards,
       CAST(COALESCE(SUM(rm.sendings_off), 0) AS INTEGER)               AS sendings_off,
       ROUND(AVG(rm.cards), 2)                                          AS cards_per_match,
       ROUND(AVG(rm.cards) - (SELECT AVG(cards) FROM ref_matches), 2)   AS above_average
FROM ref_matches rm
JOIN referees r ON r.referee_id = rm.referee_id
GROUP BY r.referee_id, r.given_name, r.family_name, r.country_name
HAVING COUNT(*) >= 5
ORDER BY cards_per_match DESC, matches DESC, referee;
"""
df = pd.read_sql_query(q22, engine)
R["Q22"] = df
print(df.head(10))

                 referee    country  matches  cards  sendings_off  cards_per_match  above_average
0           Ľuboš Micheľ   Slovakia        5     40             2             8.00           4.48
1            Howard Webb    England        6     40             1             6.67           3.14
2   Arturo Brizio Carter     Mexico        6     39             7             6.50           2.98
3        Jamal Al-Sharif      Syria        6     35             4             5.83           2.31
4       Horacio Elizondo  Argentina        5     29             3             5.80           2.28
5  Carlos Alberto Batres  Guatemala        5     27             3             5.40           1.88
6   Carlos Eugênio Simon     Brazil        7     36             1             5.14           1.62
7     Frank De Bleeckere    Belgium        7     35             1             5.00           1.48
8           Joël Quiniou     France        8     39             5             4.88           1.35
9             Óscar 

**Interpretation.** Ľuboš Micheľ (Slovakia) showed 8.0 cards per match, 4.5 above the average for all matches since 1970, followed by Howard Webb (6.67) and Arturo Brizio Carter (6.50, with 7 sendings-off). The most lenient referees with at least five matches averaged about 2 cards. Referees' bodies can use such tables to check consistency, although a referee's average also reflects the matches they were given.

### Q23. Cards per match over time, with a moving average

**Business question:** How has the number of cards per match changed since cards were introduced in 1970? Smooth the trend with a three-tournament moving average.

**SQL concepts:** CTE, LEFT JOIN, COUNT(DISTINCT ...), window function AVG() OVER (ORDER BY year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)

**Expected result:** one row per tournament 1970-2018 (13 rows, oldest first): year, matches, cards, sendings_off, cards_per_match, moving_avg_3. Peak in 2006.

In [24]:
q23 = """
WITH per AS (
    SELECT tr.year,
           COUNT(DISTINCT m.match_id)                                            AS matches,
           COUNT(b.booking_id)                                                   AS cards,
           SUM(CASE WHEN b.red_card OR b.second_yellow_card THEN 1 ELSE 0 END)   AS sendings_off
    FROM matches m
    JOIN tournaments tr  ON tr.tournament_id = m.tournament_id
    LEFT JOIN bookings b ON b.match_id = m.match_id
    WHERE tr.year >= 1970
    GROUP BY tr.year
)
SELECT year, matches, cards, sendings_off,
       ROUND(1.0 * cards / matches, 2) AS cards_per_match,
       ROUND(AVG(1.0 * cards / matches) OVER (ORDER BY year
             ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3
FROM per
ORDER BY year;
"""
df = pd.read_sql_query(q23, engine)
R["Q23"] = df
print(df.head(10))

   year  matches  cards  sendings_off  cards_per_match  moving_avg_3
0  1970       32     52             0             1.63          1.63
1  1974       38     92             5             2.42          2.02
2  1978       38     49             3             1.29          1.78
3  1982       52    104             5             2.00          1.90
4  1986       52    145             8             2.79          2.03
5  1990       52    184            16             3.54          2.78
6  1994       52    243            15             4.67          3.67
7  1998       64    276            22             4.31          4.17
8  2002       64    282            16             4.41          4.46
9  2006       64    353            27             5.52          4.74


**Interpretation.** Cards per match rose from 1.63 in 1970 to a peak of 5.52 in 2006, and the moving average shows a steady climb through the 1990s. Since then the figure has fallen to 3.52 in 2018, with only 4 sendings-off. Rule makers would want to explain that drop, for example through refereeing guidance or VAR.

### Q24. Discipline by stage: group matches vs knockout matches (1970 onwards)

**Business question:** Are knockout matches more ill-tempered than group matches? Compare cards and sendings-off per match by stage.

**SQL concepts:** CTE, LEFT JOIN (matches without cards count as 0), CASE classification, GROUP BY

**Expected result:** one row per stage played since 1970 (7 rows): phase (group/knockout), stage_name, matches, cards_per_match, sendings_off_per_match, pct_matches_with_sending_off.

In [25]:
q24 = """
WITH per_match AS (
    SELECT m.match_id, st.stage_name,
           CASE WHEN st.stage_name IN ('group stage', 'second group stage', 'final round')
                THEN 'group' ELSE 'knockout' END                                  AS phase,
           COUNT(b.booking_id)                                                   AS cards,
           SUM(CASE WHEN b.red_card OR b.second_yellow_card THEN 1 ELSE 0 END)   AS sendings_off
    FROM matches m
    JOIN tournaments tr       ON tr.tournament_id = m.tournament_id
    JOIN tournament_stages st ON st.tournament_id = m.tournament_id AND st.stage_number = m.stage_number
    LEFT JOIN bookings b      ON b.match_id = m.match_id
    WHERE tr.year >= 1970
    GROUP BY m.match_id, st.stage_name
)
SELECT phase, stage_name,
       COUNT(*)                                        AS matches,
       ROUND(AVG(cards), 2)                            AS cards_per_match,
       ROUND(AVG(COALESCE(sendings_off, 0)), 3)        AS sendings_off_per_match,
       ROUND(100.0 * SUM(CASE WHEN sendings_off > 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                       AS pct_matches_with_sending_off
FROM per_match
GROUP BY phase, stage_name
ORDER BY phase, cards_per_match DESC;
"""
df = pd.read_sql_query(q24, engine)
R["Q24"] = df
print(df.head(10))

      phase          stage_name  matches  cards_per_match  sendings_off_per_match  pct_matches_with_sending_off
0     group         group stage      504             3.37                   0.202                          17.1
1     group  second group stage       36             2.06                   0.111                          11.1
2  knockout               final       13             5.00                   0.385                          30.8
3  knockout         round of 16       72             4.67                   0.306                          22.2
4  knockout      quarter-finals       40             4.43                   0.300                          25.0
5  knockout         semi-finals       22             3.55                   0.136                          13.6
6  knockout   third-place match       13             3.00                   0.000                           0.0


**Interpretation.** Knockout matches are more heated. Finals since 1970 average 5.0 cards, and 30.8% of them had a sending-off, against 3.37 cards and 17.1% for group-stage matches. Third-place matches, with less at stake, had no sendings-off at all.

### Q25. Most-carded teams (1970 onwards)

**Business question:** Which teams have received the most cards and sendings-off, relative to how many matches they played?

**SQL concepts:** JOIN, GROUP BY, HAVING, correlated scalar subquery for matches played since 1970

**Expected result:** one row per team with at least 30 cards (33 rows), most cards first: team_name, matches, cards, yellow_only, sendings_off, cards_per_match.

In [26]:
q25 = """
SELECT t.team_name,
       (SELECT COUNT(*) FROM team_appearances ta
        JOIN tournaments tr ON tr.tournament_id = ta.tournament_id
        WHERE ta.team_id = t.team_id AND tr.year >= 1970)                                 AS matches,
       COUNT(*)                                                                         AS cards,
       SUM(CASE WHEN NOT b.red_card AND NOT b.second_yellow_card THEN 1 ELSE 0 END)     AS yellow_only,
       SUM(CASE WHEN b.red_card OR b.second_yellow_card THEN 1 ELSE 0 END)              AS sendings_off,
       ROUND(1.0 * COUNT(*) / (SELECT COUNT(*) FROM team_appearances ta
                               JOIN tournaments tr ON tr.tournament_id = ta.tournament_id
                               WHERE ta.team_id = t.team_id AND tr.year >= 1970), 2)    AS cards_per_match
FROM bookings b
JOIN teams t ON t.team_id = b.team_id
GROUP BY t.team_id, t.team_name
HAVING COUNT(*) >= 30
ORDER BY cards DESC;
"""
df = pd.read_sql_query(q25, engine)
R["Q25"] = df
print(df.head(10))

     team_name  matches  cards  yellow_only  sendings_off  cards_per_match
0    Argentina       65    123          117             6             1.89
1       Brazil       77    102           96             6             1.32
2  Netherlands       48    101           94             7             2.10
3        Italy       63     98           92             6             1.56
4       Mexico       40     85           79             6             2.13
5      Germany       41     79           75             4             1.93
6       France       49     75           69             6             1.53
7  South Korea       32     74           72             2             2.31
8      Uruguay       36     68           62             6             1.89
9        Spain       48     66           65             1             1.38


**Interpretation.** Argentina have received the most cards since 1970 (123 in 65 matches). Relative to matches played, Ghana (2.92 per match), Chile (2.50) and Cameroon (2.39, with 8 sendings-off) are the most carded. Disciplinary committees and coaches should compare cards per match rather than raw totals, because teams played very different numbers of matches.

## 5. Venues and hosting

### Q26. Busiest stadiums

**Business question:** Which stadiums have hosted the most World Cup matches, and how many finals?

**SQL concepts:** multi-table JOIN, GROUP BY, window function RANK() in a derived table, filtering on the rank (ties kept), conditional aggregation

**Expected result:** every stadium ranked 10th or better by matches, ties included: rank, stadium, city, country, capacity, matches, tournaments, finals, first_year, last_year. Estadio Azteca first with 19 matches.

In [27]:
q26 = """
SELECT *
FROM (
SELECT RANK() OVER (ORDER BY COUNT(*) DESC)                      AS rank,
       s.stadium_name, s.city_name, s.country_name,
       s.stadium_capacity                                       AS capacity,
       COUNT(*)                                                 AS matches,
       COUNT(DISTINCT m.tournament_id)                          AS tournaments,
       SUM(CASE WHEN st.stage_name = 'final' THEN 1 ELSE 0 END) AS finals,
       MIN(tr.year)                                             AS first_year,
       MAX(tr.year)                                             AS last_year
FROM matches m
JOIN stadiums s            ON s.stadium_id = m.stadium_id
JOIN tournaments tr        ON tr.tournament_id = m.tournament_id
JOIN tournament_stages st  ON st.tournament_id = m.tournament_id AND st.stage_number = m.stage_number
GROUP BY s.stadium_id, s.stadium_name, s.city_name, s.country_name, s.stadium_capacity
) ranked
WHERE rank <= 10
ORDER BY rank, finals DESC, stadium_name;
"""
df = pd.read_sql_query(q26, engine)
R["Q26"] = df
print(df.head(10))

   rank         stadium_name       city_name country_name  capacity  matches  tournaments  finals  first_year  last_year
0     1       Estadio Azteca     Mexico City       Mexico    115000       19            2       2        1970       1986
1     2  Estádio do Maracanã  Rio de Janeiro       Brazil    200000       15            2       1        1950       2014
2     3      Estadio Jalisco     Guadalajara       Mexico     71000       14            2       0        1970       1986
3     4     Estadio Nou Camp            León       Mexico     31000       11            2       0        1970       1986
4     5   Estadio Centenario      Montevideo      Uruguay     90000       10            1       1        1930       1930
5     5     Estadio Nacional        Santiago        Chile     67000       10            1       1        1962       1962
6     5        Neckarstadion       Stuttgart      Germany     72000       10            2       0        1974       2006
7     5          Waldstadion    

**Interpretation.** Estadio Azteca has hosted the most World Cup matches (19), including two finals, followed by the Maracanã (15); nine stadiums share tenth place with 9 matches, so the list has 18 rows. Mexican stadiums dominate the list because Mexico hosted twice, in 1970 and 1986, which is relevant for host-bid committees planning to reuse venues.

## 6. Managers

### Q27. Most experienced and most successful managers

**Business question:** Which managers have managed the most World Cup matches, and how many did they win?

**SQL concepts:** JOIN on a composite key (match_id, team_id), multi-table JOIN, GROUP BY, STRING_AGG(DISTINCT ...), conditional aggregation, RANK() in a derived table (ties kept)

**Expected result:** every manager ranked 10th or better by matches, ties included: manager, nationality, teams, tournaments, matches, won, drawn, lost, win_pct. Helmut Schoen first (25 matches, 16 wins).

In [28]:
q27 = """
SELECT manager, nationality, teams, tournaments, matches, won, drawn, lost, win_pct
FROM (
SELECT RANK() OVER (ORDER BY COUNT(*) DESC)                                 AS rank,
       COALESCE(mg.given_name || ' ', '') || mg.family_name                 AS manager,
       mg.country_name                                                      AS nationality,
       STRING_AGG(DISTINCT t.team_name, ', ')                               AS teams,
       COUNT(DISTINCT ma.tournament_id)                                     AS tournaments,
       COUNT(*)                                                             AS matches,
       SUM(CASE WHEN ta.goals_for > ta.goals_against THEN 1 ELSE 0 END)     AS won,
       SUM(CASE WHEN ta.goals_for = ta.goals_against THEN 1 ELSE 0 END)     AS drawn,
       SUM(CASE WHEN ta.goals_for < ta.goals_against THEN 1 ELSE 0 END)     AS lost,
       ROUND(100.0 * SUM(CASE WHEN ta.goals_for > ta.goals_against THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                                            AS win_pct
FROM manager_appearances ma
JOIN team_appearances ta ON ta.match_id = ma.match_id AND ta.team_id = ma.team_id
JOIN managers mg         ON mg.manager_id = ma.manager_id
JOIN teams t             ON t.team_id = ma.team_id
GROUP BY mg.manager_id, mg.given_name, mg.family_name, mg.country_name
) ranked
WHERE rank <= 10
ORDER BY matches DESC, won DESC, manager;
"""
df = pd.read_sql_query(q27, engine)
R["Q27"] = df
print(df.head(10))

                   manager  nationality                                                        teams  tournaments  matches  won  drawn  lost  win_pct
0             Helmut Schön      Germany                                                 West Germany            4       25   16      5     4     64.0
1  Carlos Alberto Parreira       Brazil  Brazil, Kuwait, Saudi Arabia, South Africa, United Arab ...            6       24   10      5     9     41.7
2      Luiz Felipe Scolari       Brazil                                             Brazil, Portugal            3       21   14      3     4     66.7
3            Mário Zagallo       Brazil                                                       Brazil            3       20   13      3     4     65.0
4            Óscar Tabárez      Uruguay                                                      Uruguay            4       20   10      3     7     50.0
5         Bora Milutinović   Yugoslavia            China, Costa Rica, Mexico, Nigeria, United States

**Interpretation.** Helmut Schön (West Germany) managed the most matches (25) and won 16 of them. Bora Milutinović and Carlos Alberto Parreira each took five different countries to World Cups, and Joachim Löw has the best win rate in this list (70.6%).

### Q28. Foreign managers vs home-nation managers

**Business question:** Do teams led by a foreign manager go further than teams led by a manager of their own nationality?

**SQL concepts:** CTEs, CASE (classification, including a historical-name rule), MIN/MAX of a flag to classify joint managers, LEFT JOIN, correlated subquery, AVG

**Expected result:** one row per manager type (2 rows on this data: home-nation, foreign): manager_type, team_tournaments, avg_matches_per_tournament, pct_reached_top_four, titles. A 'mixed' type appears only if joint managers differ in nationality; in the source every joint pair shares one nationality (even 1998 Saudi Arabia and Tunisia, where both managers are listed as Brazil / Poland). Rule: a manager counts as home-nation when nationality = team name, or for West Germany when nationality = 'Germany' (the source lists West German managers as 'Germany').

In [29]:
q28 = """
WITH appointment_flags AS (
    SELECT a.tournament_id, a.team_id,
           CASE WHEN m.country_name = t.team_name
                  OR (t.team_name = 'West Germany' AND m.country_name = 'Germany')
                THEN 0 ELSE 1 END AS is_foreign
    FROM manager_appointments a
    JOIN managers m ON m.manager_id = a.manager_id
    JOIN teams t    ON t.team_id = a.team_id
),
team_tournaments AS (
    SELECT tournament_id, team_id,
           CASE WHEN MIN(is_foreign) = 1 THEN 'foreign'
                WHEN MAX(is_foreign) = 0 THEN 'home-nation'
                ELSE 'mixed (joint managers)' END AS manager_type
    FROM appointment_flags
    GROUP BY tournament_id, team_id
)
SELECT tt.manager_type,
       COUNT(*)                                                                  AS team_tournaments,
       ROUND(AVG((SELECT COUNT(*) FROM team_appearances ta
                  WHERE ta.tournament_id = tt.tournament_id AND ta.team_id = tt.team_id)), 2)
                                                                                 AS avg_matches_per_tournament,
       ROUND(100.0 * SUM(CASE WHEN s.position IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                                                 AS pct_reached_top_four,
       SUM(CASE WHEN s.position = 1 THEN 1 ELSE 0 END)                           AS titles
FROM team_tournaments tt
LEFT JOIN tournament_standings s
       ON s.tournament_id = tt.tournament_id AND s.team_id = tt.team_id
GROUP BY tt.manager_type
ORDER BY team_tournaments DESC;
"""
df = pd.read_sql_query(q28, engine)
R["Q28"] = df
print(df.head(10))

  manager_type  team_tournaments  avg_matches_per_tournament  pct_reached_top_four  titles
0  home-nation               331                        4.12                  22.7      21
1      foreign               126                        3.45                   7.1       0


**Interpretation.** Teams with a home-nation manager reached the top four 22.7% of the time and won all 21 titles; teams with a foreign manager reached the top four only 7.1% of the time and never won. This is a correlation, not a cause: strong football nations usually appoint their own coaches, while foreign coaches are more often hired by less established teams.

## 7. Awards

### Q29. Does the best player (Golden Ball) come from the champion team?

**Business question:** How often does the Golden Ball go to a player from the team that won the tournament, and how did the winners' teams finish?

**SQL concepts:** multi-table JOIN, LEFT JOIN with IS NULL handling, CASE

**Expected result:** one row per Golden Ball winner (11 rows, 1978-2018, newest first): year, player, team, team_finish ('champion', 'runner-up', '3rd', '4th' or 'outside top four').

In [30]:
q29 = """
SELECT tr.year,
       COALESCE(p.given_name || ' ', '') || p.family_name AS player,
       t.team_name                                        AS team,
       CASE s.position
            WHEN 1 THEN 'champion'
            WHEN 2 THEN 'runner-up'
            WHEN 3 THEN '3rd'
            WHEN 4 THEN '4th'
            ELSE 'outside top four'
       END                                                AS team_finish
FROM award_winners aw
JOIN awards a       ON a.award_id = aw.award_id
JOIN players p      ON p.player_id = aw.player_id
JOIN teams t        ON t.team_id = aw.team_id
JOIN tournaments tr ON tr.tournament_id = aw.tournament_id
LEFT JOIN tournament_standings s
       ON s.tournament_id = aw.tournament_id AND s.team_id = aw.team_id
WHERE a.award_name = 'Golden Ball'
ORDER BY tr.year DESC;
"""
df = pd.read_sql_query(q29, engine)
R["Q29"] = df
print(df.head(10))

   year               player       team team_finish
0  2018          Luka Modrić    Croatia   runner-up
1  2014         Lionel Messi  Argentina   runner-up
2  2010         Diego Forlán    Uruguay         4th
3  2006      Zinedine Zidane     France   runner-up
4  2002          Oliver Kahn    Germany   runner-up
5  1998              Ronaldo     Brazil   runner-up
6  1994              Romário     Brazil    champion
7  1990  Salvatore Schillaci      Italy         3rd
8  1986       Diego Maradona  Argentina    champion
9  1982          Paolo Rossi      Italy    champion


**Interpretation.** Only 4 of the 11 Golden Ball winners came from the champion team (1978, 1982, 1986, 1994). Since 1998 the award has always gone to a player from a beaten team, usually the runner-up. The award recognises individual performance more than team success.

## 8. Data quality and integrity

### Q30. Data integrity: do the goal records match the scores?

**Business question:** Does every goal in the goals table agree with the match scores? (A check that the database is complete and consistent after loading.)

**SQL concepts:** CTEs, LEFT JOIN with COALESCE (team-matches without goals count 0), set comparison with EXCEPT, NOT EXISTS anti-join, scalar subqueries

**Expected result:** exactly 1 summary row: team_matches_checked (1800), goals_in_scores (2548), goal_rows (2548), mismatched_team_matches (0), goal_groups_without_match_team (0).

In [31]:
q30 = """
WITH goal_rows AS (
    SELECT match_id, team_id, COUNT(*) AS goals
    FROM goals
    GROUP BY match_id, team_id
),
compared AS (
    SELECT ta.match_id, ta.team_id,
           ta.goals_for               AS goals_in_score,
           COALESCE(g.goals, 0)       AS goals_in_goal_rows
    FROM team_appearances ta
    LEFT JOIN goal_rows g ON g.match_id = ta.match_id AND g.team_id = ta.team_id
),
mismatches AS (
    SELECT match_id, team_id, goals_in_goal_rows FROM compared
    EXCEPT
    SELECT match_id, team_id, goals_in_score     FROM compared
)
SELECT (SELECT COUNT(*) FROM compared)                       AS team_matches_checked,
       (SELECT CAST(SUM(goals_in_score) AS INTEGER) FROM compared)     AS goals_in_scores,
       (SELECT CAST(SUM(goals_in_goal_rows) AS INTEGER) FROM compared) AS goal_rows,
       (SELECT COUNT(*) FROM mismatches)                     AS mismatched_team_matches,
       (SELECT COUNT(*) FROM goal_rows g
        WHERE NOT EXISTS (SELECT 1 FROM team_appearances ta
                          WHERE ta.match_id = g.match_id AND ta.team_id = g.team_id))
                                                             AS goal_groups_without_match_team;
"""
df = pd.read_sql_query(q30, engine)
R["Q30"] = df
print(df.head(10))

   team_matches_checked  goals_in_scores  goal_rows  mismatched_team_matches  goal_groups_without_match_team
0                  1800             2548       2548                        0                               0

**Interpretation.** The 2,548 goal rows reproduce all 1,800 team-match scores exactly, with no mismatches and no goal attached to a team that was not in the match. The load was complete and consistent, so every goal-based query above can be trusted.

## 9. Validation of results

Each check below compares a query result with **(a)** a calculation done independently in pandas directly on the raw CSV files, not through the database or the cleaning code, and, where one exists, **(b)** a well-known World Cup fact. A check passes when all available values agree. Differences that come from a deliberate convention are marked *explained* and discussed after the table.

In [32]:
raw = {n: pd.read_csv(ROOT / "data" / f"{n}.csv", keep_default_na=False, na_values=[""])
       for n in ["tournaments", "matches", "goals", "team_appearances", "penalty_kicks", "bookings",
                 "manager_appearances", "player_appearances", "players", "award_winners", "host_countries"]}
g, ta, m = raw["goals"], raw["team_appearances"], raw["matches"]
real = g[g.own_goal == 0]
checks = []
def check(cid, query, what, sql_value, pandas_value, known=None, explained=False):
    vals = [v for v in (pandas_value, known) if v is not None]
    status = "match" if all(str(v) == str(sql_value) for v in vals) else ("explained" if explained else "MISMATCH")
    checks.append((cid, query, what, sql_value, pandas_value, known, status))

# V1 titles
check("V01", "Q01", "Brazil titles", int(R["Q01"].set_index("team_name").loc["Brazil", "titles"]),
      int((raw["tournaments"].winner == "Brazil").sum()), 5)
# V2 Brazil record (scores; shoot-out = draw)
b = ta[ta.team_name == "Brazil"]
sql_b = R["Q02"].set_index("team_name").loc["Brazil"]
check("V02", "Q02", "Brazil played / won / drawn / lost / goals for-against",
      "/".join(str(int(sql_b[c])) for c in ["played", "won", "drawn", "lost"]) + f"/{int(sql_b.goals_for)}-{int(sql_b.goals_against)}",
      f"{len(b)}/{(b.goals_for > b.goals_against).sum()}/{(b.goals_for == b.goals_against).sum()}/{(b.goals_for < b.goals_against).sum()}/{b.goals_for.sum()}-{b.goals_against.sum()}",
      "109/73/18/18/229-105")
# V3 host win rate
hosts = set(map(tuple, raw["host_countries"][["tournament_id", "team_id"]].values))
is_host = [(t, x) in hosts for t, x in zip(ta.tournament_id, ta.team_id)]
hw = ta[is_host]
check("V03", "Q03", "host win % (all years)",
      float(R["Q03"].query("side == 'host' and era == 'all years'").win_pct.iloc[0]),
      round(100 * (hw.goals_for > hw.goals_against).mean(), 1))
# V4 Klose
check("V04", "Q10", "Miroslav Klose goals", int(R["Q10"].set_index("player").loc["Miroslav Klose", "goals"]),
      int((real.family_name == "Klose").sum()), 16)
# V5 Fontaine 1958
check("V05", "Q11", "top scorer goals 1958 (Just Fontaine)", int(R["Q11"].set_index("year").loc[1958, "goals"]),
      int(real[real.tournament_id == "WC-1958"].groupby("player_id").size().max()), 13)
# V6 hat-tricks and record
per = real.groupby(["match_id", "player_id"]).size()
check("V06", "Q12", "number of hat-tricks / most goals in a match", f"{len(R['Q12'])}/{R['Q12'].goals.max()}",
      f"{(per >= 3).sum()}/{per.max()}", None)
check("V07", "Q12", "Oleg Salenko goals in one match", int(R["Q12"].iloc[0].goals),
      int(per.max()), 5)
# V8 total goals and 1954 peak
q17 = R["Q17"].set_index("year")
check("V08", "Q17", "total goals 1930-2018", int(q17.goals.sum()),
      int((m.home_team_score + m.away_team_score).sum()), 2548)
check("V09", "Q17", "goals per match 1954", float(q17.loc[1954, "goals_per_match"]),
      round((m[m.tournament_id == "WC-1954"].home_team_score + m[m.tournament_id == "WC-1954"].away_team_score).mean(), 2), 5.38)
# V10 biggest win
mm = m.assign(margin=(m.home_team_score - m.away_team_score).abs())
big = mm.sort_values(["margin", "home_team_score"], ascending=False).iloc[0]
check("V10", "Q18", "biggest win", R["Q18"].iloc[0]["match"],
      f"{big.home_team_name} {big.home_team_score}-{big.away_team_score} {big.away_team_name}",
      "Hungary 10-1 El Salvador")
# V11 Argentina shoot-outs
so = ta[(ta.penalty_shootout == 1) & (ta.team_name == "Argentina")]
q21 = R["Q21"].set_index("team_name").loc["Argentina"]
check("V11", "Q21", "Argentina shoot-outs won / played", f"{int(q21.won)}/{int(q21.shootouts)}",
      f"{(so.penalties_for > so.penalties_against).sum()}/{len(so)}", "4/5")
# V12 Matthaeus appearances (person level vs per team)
pa = raw["player_appearances"]
mat = pa[pa.family_name == "Matthäus"]
q15 = R["Q15"]
check("V12", "Q15", "Lothar Matthäus appearances (all teams)",
      f"not top for either team; top: Germany {q15.set_index('team_name').loc['Germany','appearances']}, West Germany {q15.set_index('team_name').loc['West Germany','appearances']}",
      f"{len(mat)} ({', '.join(f'{k} {v}' for k, v in mat.team_name.value_counts().items())})", 25, explained=True)
# V13 substitutes 2014
sub = real.merge(pa[["match_id", "player_id", "substitute"]], on=["match_id", "player_id"])
check("V13", "Q13", "goals by substitutes 2014", int(R["Q13"].set_index("year").loc[2014, "substitute_goals"]),
      int(sub[(sub.tournament_id == "WC-2014") & (sub.substitute == 1)].shape[0]), 32)
# V14 cards 2006
check("V14", "Q23", "card events 2006", int(R["Q23"].set_index("year").loc[2006, "cards"]),
      int((raw["bookings"].tournament_id == "WC-2006").sum()))
# V15 Azteca
check("V15", "Q26", "Estadio Azteca matches", int(R["Q26"].set_index("stadium_name").loc["Estadio Azteca", "matches"]),
      int((m.stadium_name == "Estadio Azteca").sum()), 19)
# V16 Schoen
ma = raw["manager_appearances"]
sch = ma[ma.family_name == "Schön"].merge(ta[["match_id", "team_id", "goals_for", "goals_against"]], on=["match_id", "team_id"])
q27 = R["Q27"].set_index("manager").loc["Helmut Schön"]
check("V16", "Q27", "Helmut Schön matches / wins", f"{int(q27.matches)}/{int(q27.won)}",
      f"{len(sch)}/{(sch.goals_for > sch.goals_against).sum()}", "25/16")
# V17 integrity
check("V17", "Q30", "goal rows = goals in scores", f"{R['Q30'].goal_rows[0]}={R['Q30'].goals_in_scores[0]}",
      f"{len(g)}={int((m.home_team_score + m.away_team_score).sum())}", "2548=2548")
# V18 Golden Boot 2010
q11 = R["Q11"].set_index("year")
aw = raw["award_winners"]
gb = aw[(aw.tournament_id == "WC-2010") & (aw.award_name == "Golden Boot")]
check("V18", "Q11", "2010 top scorer(s) vs Golden Boot", f"{q11.loc[2010, 'top_scorers']} -> boot: {q11.loc[2010, 'golden_boot']}",
      f"boot in award_winners.csv: {' '.join(gb.given_name + ' ' + gb.family_name)}", None, explained=True)

val = pd.DataFrame(checks, columns=["check", "query", "what is compared", "SQL result", "pandas on raw CSV", "known fact", "status"])
print(val.to_string(index=False))
print()
print(val.status.value_counts().to_string())

check query                                       what is compared                                                                       SQL result                        pandas on raw CSV               known fact    status
  V01   Q01                                          Brazil titles                                                                                5                                        5                        5     match
  V02   Q02 Brazil played / won / drawn / lost / goals for-against                                                             109/73/18/18/229-105                     109/73/18/18/229-105     109/73/18/18/229-105     match
  V03   Q03                                 host win % (all years)                                                                             62.7                                     62.7                     None     match
  V04   Q10                                   Miroslav Klose goals                                      

In [33]:
# The 'result' column trap (discussed below): Brazil's wins if the raw 0/1 'win' flag were used
print("Brazil wins from the raw 'win' flag :", int(ta[ta.team_name == "Brazil"].win.sum()))
print("Brazil wins from goals (our method) :", int((b.goals_for > b.goals_against).sum()))
print("Brazil shoot-out wins counted by the raw flag:",
      int(((b.penalty_shootout == 1) & (b.win == 1)).sum()))

Brazil wins from the raw 'win' flag : 76
Brazil wins from goals (our method) : 73
Brazil shoot-out wins counted by the raw flag: 3


### Mismatches found and how they were resolved

- **V01-V11, V13-V17: match.** Every SQL result equals the pandas calculation on the raw CSV files and, where one exists, the well-known fact. Examples: Brazil's 5 titles, Klose's 16 goals, Fontaine's 13 in 1958, Salenko's 5 in one match, 2,548 goals in total, Hungary 10-1 El Salvador, Argentina's 4 shoot-out wins, 32 substitute goals in 2014, 19 matches at the Azteca, and Helmut Schön's 25 matches and 16 wins.
- **V02, a mismatch avoided by design.** The raw `result`/`win` columns count a shoot-out win as a win. Using them would give Brazil 76 wins instead of FIFA's 73 (the cell above shows the 3 shoot-out wins involved). Our queries take results from the score, so a shoot-out is a draw, and the SQL then agrees with the official record of 109 matches, 73 wins, 18 draws and 18 losses.
- **V12, explained.** Lothar Matthäus holds the appearance record with 25 matches, and pandas confirms 25 in the raw data, but Q15 shows other players on top. His appearances are split between West Germany (16) and Germany (9), and Q15 counts teams by `team_id`. This is the documented convention for historical teams, not an error. A person-level query, grouping by player only, returns his 25.
- **V18, explained.** In 2010 four players scored 5 goals, but the Golden Boot went to Thomas Müller alone. FIFA broke the tie on a further criterion (assists) that is not in this dataset, so Q11 correctly reports "different". The award data itself, from `award_winners.csv`, agrees with the SQL.
- **Data observation from Q28 (no fix applied).** The source gives both joint managers of 1998 Saudi Arabia the nationality "Brazil", and both joint managers of 1998 Tunisia "Poland". The local co-managers' names suggest the source copied the head coach's nationality. We report this but do not change the data, because the dataset is our only source; as a result, Q28 has no "mixed" category.

## 10. Summary

- **Queries:** all 30 ran successfully against the database built in Milestones 2 and 3. Together they cover inner, outer and self-joins, GROUP BY/HAVING, scalar, correlated and EXISTS subqueries, CTEs, window functions (RANK, DENSE_RANK, ROW_NUMBER, LAG, running totals, moving averages, framed averages), set operations (EXCEPT, UNION ALL), CASE logic, conditional aggregation, date arithmetic and string aggregation.
- **Validation:** 18 checks were made against independent pandas calculations and well-known facts. All agree, except two differences that come from documented conventions (V12, V18).
- **Key findings for users:**
  - Brazil's dominance: 5 titles and 67% of matches won.
  - Hosts win 63% of their matches, but less often than they used to.
  - Scoring first is decisive.
  - Substitutes and late goals matter more and more.
  - Cards peaked in 2006.
  - VAR coincided with a record share of penalty goals in 2018.